# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook Colab (GPU T4) chạy lại toàn bộ bài lab. Mọi thí nghiệm đi qua **một** hàm `train.run(Config)`;
các stage được điều phối bởi `run_experiments.py` (chạy tiếp được nếu phiên bị ngắt: lần chạy đã có
`summary.json` sẽ được bỏ qua). Kết quả ghi thẳng vào thư mục bài nộp trên Google Drive:
`runs/` (checkpoint, log, logit — không commit), `predictions/`, `curves/`, `figures/`, `tables/`, `results.xlsx`.

**Quy tắc:** mọi lựa chọn dựa trên **val**; test fold 0 chỉ được đánh giá ở Bước 4, **một lần mỗi seed**
(`train.evaluate_split` từ chối chạy lại nếu logit test đã tồn tại).

## 0. Môi trường và dữ liệu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess
REPO = '/content/drive/MyDrive/VinAI/K4-Track4-Day2-Deeplearning-Advance'   # đổi nếu repo ở chỗ khác
SUB = REPO + '/submissions/MSSV_ho_ten'; CODE = SUB + '/code'
os.environ.update(LAB_SUB=SUB, LAB_DATA='/content/data', LAB_EPOCHS='10', PYTHONUNBUFFERED='1')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip list 2>/dev/null | grep -i -E "^(torch|torchvision|timm|numpy|pandas|openpyxl|matplotlib) "

In [ ]:
# Ảnh (Zenodo, kiểm tra MD5) + nhãn fold 0 (GitHub tác giả). Lưu bản sao zip trên Drive cho phiên sau.
import hashlib, shutil
DATA = '/content/data'; os.makedirs(f'{DATA}/labels', exist_ok=True)
ZIP = f'{DATA}/images.zip'; ZIP_DRIVE = '/content/drive/MyDrive/VinAI/deepweeds_images.zip'
if not os.path.exists(ZIP):
    if os.path.exists(ZIP_DRIVE):
        shutil.copy(ZIP_DRIVE, ZIP)
    else:
        !wget -q -O {ZIP} "https://zenodo.org/records/7939060/files/images.zip?download=1"
h = hashlib.md5()
with open(ZIP, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
assert h.hexdigest() == 'b7b30f96d466fba86016aa5a26606e0f', f'MD5 sai: {h.hexdigest()}'
if not os.path.exists(ZIP_DRIVE): shutil.copy(ZIP, ZIP_DRIVE)
!unzip -q -n {ZIP} -d {DATA}/images
BASE = 'https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels'
for name in ['labels', 'train_subset0', 'val_subset0', 'test_subset0']:
    !wget -q -O {DATA}/labels/{name}.csv {BASE}/{name}.csv
!ls {DATA}/images | wc -l; wc -l {DATA}/labels/*.csv

In [ ]:
%cd {CODE}
!python -m unittest test_code 2>&1 | tail -4      # kiểm tra tự viết: focal γ=0 ≡ CE, CutMix, gộp BN, EMA, ...
from IPython.display import Image, display
import pandas as pd, json
def run(*args):   # chạy một stage, in log ra notebook
    !python run_experiments.py {' '.join(args)} 2>&1 | grep -v -i warn

## Bước 0 — EDA và kiểm tra pipeline (README 2.1, GUIDE 1.2–1.3)

In [ ]:
run('--stage eda')          # kiểm tra chia (giao rỗng, hợp 17.509, file tồn tại), biểu đồ, ảnh mẫu, cache .npy
display(pd.read_csv(f'{SUB}/tables/eda_per_class.csv', index_col=0))
display(Image(f'{SUB}/figures/eda_class_distribution.png')); display(Image(f'{SUB}/figures/eda_samples.png', width=500))

In [ ]:
run('--stage sanity --backbone resnet50')   # loss ban đầu ≈ ln 9, overfit 16 ảnh, ảnh sau augmentation
print({k: v for k, v in json.load(open(f'{SUB}/tables/sanity_checks.json')).items() if k != 'overfit_curve'})
for f in ('sanity_overfit', 'aug_check_basic', 'aug_check_cutmix'): display(Image(f'{SUB}/figures/{f}.png', width=700))

## Bước 1 — So sánh 5 backbone (công thức nền T00, seed 0, 10 epoch)
B01 ResNet-50 · B02 ConvNeXt-T · B03 DeiT-S (transformer) · B04 EfficientNet-B0 (nhẹ) · B05 MobileNetV3-L (nhẹ).

In [ ]:
run('--stage backbones')
rows = [json.load(open(f'{SUB}/runs/{e}/seed0/summary.json')) for e in ('B01','B02','B03','B04','B05')]
pd.DataFrame(rows)[['exp_id','backbone','weight_tag','params_M','gmacs','val_macro_f1','val_top1',
                    'train_time_per_epoch_s','latency_b1_fp32_p50_ms']]

## Bước 2 — Công thức huấn luyện trên `convnext_tiny` (mỗi lần chạy khác T00 đúng một yếu tố)
T00 seed0 trùng cấu hình với lần chạy backbone tương ứng nên được dùng lại (ghi `alias_of`).
Sau đó chạy **một kết hợp** (T10) các yếu tố thắng trên val.

In [ ]:
BACKBONE = 'convnext_tiny'
run('--stage training --backbone', BACKBONE)

In [ ]:
COMBO = 'mix=cutmix loss=focal ema_decay=0.999'   # các yếu tố tốt nhất trên val (xem report.md mục 4)
run('--stage combo --backbone', BACKBONE, f'--combo "{COMBO}"')

## Bước 3 — Suy luận (chỉ trên val) và độ trễ (warmup 10, cuda.synchronize, 100 lần, p50/p95/p99)

In [ ]:
run('--stage inference --backbone', BACKBONE)
display(pd.read_csv(f'{SUB}/tables/inference.csv')); display(pd.read_csv(f'{SUB}/tables/latency.csv'))
print(json.load(open(f'{SUB}/tables/inference_choice.json')))

## Bước 4 — Chung kết: 3 seed, test MỘT lần mỗi seed
F01 = backbone + công thức kết hợp + suy luận đã chọn + temperature scaling (T khớp trên val).
Mốc = T00 + I00 (1-view), cùng 3 seed.

In [ ]:
run('--stage final --backbone', BACKBONE, f'--combo "{COMBO}"', '--seeds 0,1,2')

In [ ]:
L = '/content/data/labels'
!python {REPO}/eval.py score --pred "{SUB}/predictions/F01_seed*_test.csv" --test-csv {L}/test_subset0.csv --labels {L}/labels.csv --tag F01 --out {SUB}/eval_out
!python {REPO}/eval.py score --pred "{SUB}/predictions/T00_seed*_test.csv" --test-csv {L}/test_subset0.csv --labels {L}/labels.csv --tag T00 --out {SUB}/eval_out

In [ ]:
lat = pd.read_csv(f'{SUB}/tables/latency.csv'); p95 = lat[(lat.batch == 1) & (lat.dtype == 'fp32') & lat.config.str.startswith('T00')].p95.iloc[0]
!python {REPO}/eval.py grade --final "{SUB}/predictions/F01_seed*_test.csv" --baseline "{SUB}/predictions/T00_seed*_test.csv" \
    --uncal "{SUB}/predictions/F01uncal_seed*_test.csv" --final-val "{SUB}/predictions/F01_seed*_val.csv" \
    --latency-p95-ms {p95:.2f} --latency-method proper --test-csv {L}/test_subset0.csv --labels {L}/labels.csv --out {SUB}/eval_out

## Bước 5 — results.xlsx và biểu đồ

In [ ]:
!python make_results.py --backbone {BACKBONE} 2>&1 | grep -v -i warn
x = pd.read_excel(f'{SUB}/results.xlsx', sheet_name=None)
for k, v in x.items(): print(k, v.shape)
display(x['Summary'])
for f in ('backbones_tradeoff', 'training_ablation', 'inference_tradeoff', 'confusion_F01', 'errors_chinee_snake'):
    display(Image(f'{SUB}/figures/{f}.png', width=800))